# Analysis of heads with diagonal patterns

The goal is to

In [ ]:
import math
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import seaborn as sns

## See the heads of a model

In [ ]:
CACHE_PATH = Path("../data/cache/chatmode/qwen25_gqa_cache.pt")
cache = torch.load(CACHE_PATH, map_location="cpu")

meta = cache["meta"]
prompts = cache["prompts"]

print("Model:", cache["model_id"])
print("Q heads:", meta["num_q_heads"], "| KV heads:", meta["num_kv_heads"], "| KV groups:", meta["num_kv_groups"])

PROMPT_IDX = 1
LAYER_IDX = 0

MAX_LABEL_LEN = 18
FIGSIZE_PER_PANEL = 6

record = prompts[PROMPT_IDX]
tokens = record["tokens"]
attn = record["attentions"][LAYER_IDX]   # [bsz, q_heads, tgt_len, src_len]
assert attn is not None, "Attention weights are missing. Ensure attn_implementation='eager'."

attn = attn[0].float()                   # [q_heads, seq_len, seq_len]
num_heads, tgt_len, src_len = attn.shape
kv_map = record["layers"][LAYER_IDX]["kv_head_for_q"].tolist()

def short_tok(t: str, max_len: int = MAX_LABEL_LEN) -> str:
    t = t.replace("Ġ", "▁").replace("Ċ", "\\n")
    return t if len(t) <= max_len else t[: max_len - 1] + "…"

labels = [short_tok(t) for t in tokens]

ncols = 2
nrows = math.ceil(num_heads / ncols)
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * FIGSIZE_PER_PANEL, nrows * FIGSIZE_PER_PANEL),
    squeeze=False
)

for h in range(nrows * ncols):
    ax = axes[h // ncols][h % ncols]
    if h >= num_heads:
        ax.axis("off")
        continue

    sns.heatmap(
        attn[h].numpy(),
        ax=ax,
        cmap="magma",
        vmin=0.0,
        vmax=float(attn.max()),
        cbar=False,
        square=True,
        xticklabels=labels,
        yticklabels=labels,
    )
    ax.set_title(f"Head q={h} → kv={kv_map[h]}")
    ax.set_xlabel("Source token")
    ax.set_ylabel("Target token")
    ax.tick_params(axis="x", rotation=90, labelsize=8)
    ax.tick_params(axis="y", rotation=0, labelsize=8)

fig.suptitle(
    f"Attention heatmaps | prompt={PROMPT_IDX} | layer={LAYER_IDX}\n"
    f"{record['text']}",
    y=1.02
)
plt.tight_layout()
plt.show()

## Explore one head deeply

In [ ]:
CACHE_PATH = Path("../data/cache/chatmode/qwen25_full_block_cache.pt")

prompt_id = 0
layer_idx = 0
head_idx = 1
batch_idx = 0

payload = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
rec = payload["prompts"][prompt_id]
layer = rec["layers"][layer_idx]
tokens = rec["tokens"]
tokens = [short_tok(t) for t in tokens]

kv_head_idx = int(layer["kv_head_for_q"][head_idx].item())

# Tenseurs à afficher
attn = layer["attn_weights"][batch_idx, head_idx]                    # [seq, seq]
head_in = layer["input_layernorm_out"][batch_idx]                    # [seq, hidden]
v_mat = layer["v_pre"][batch_idx, :, kv_head_idx]                    # [seq, head_dim]
norm1_out = layer["input_layernorm_out"][batch_idx]                  # [seq, hidden]
head_out = layer["head_out"][batch_idx, :, head_idx]                 # [seq, head_dim]
resid1 = layer["residual_after_attn"][batch_idx]                     # [seq, hidden]
norm2_out = layer["post_attn_layernorm_out"][batch_idx]              # [seq, hidden]
mlp_out = layer["mlp_out"][batch_idx]                                # [seq, hidden]
final_out = layer["block_output"][batch_idx]                         # [seq, hidden]

items = [
    ("Attention", attn, "tokens", "tokens"),
    ("Entrée tête / après 1re norm", head_in, "hidden dim", "tokens"),
    (f"Matrice V (kv head {kv_head_idx})", v_mat, "head dim", "tokens"),
    ("Sortie après 1re normalisation", norm1_out, "hidden dim", "tokens"),
    (f"Sortie après attention (head {head_idx})", head_out, "head dim", "tokens"),
    ("Après 1er residual stream", resid1, "hidden dim", "tokens"),
    ("Sortie après 2e normalisation", norm2_out, "hidden dim", "tokens"),
    ("Sortie après MLP", mlp_out, "hidden dim", "tokens"),
    ("Sortie finale après 2e residual stream", final_out, "hidden dim", "tokens"),
]

def maybe_center(name):
    return 0 if any(k in name.lower() for k in ["sortie", "résidual", "residual", "mlp", "norm", "entrée"]) else None

n = len(items)
fig, axes = plt.subplots(n, 1, figsize=(22, 10 * n), constrained_layout=True)

if n == 1:
    axes = [axes]

for ax, (title, tensor, xlabel, ylabel) in zip(axes, items):
    x = tensor.float().numpy()
    center = maybe_center(title)
    cmap = "coolwarm" if center == 0 else "viridis"

    sns.heatmap(
        x,
        ax=ax,
        cmap=cmap,
        center=center,
        cbar=True,
        xticklabels=False if x.shape[1] > 80 else True,
        yticklabels=tokens if x.shape[0] == len(tokens) and len(tokens) <= 80 else False,
    )
    ax.set_title(f"Layer {layer_idx} | Head {head_idx} — {title}", fontsize=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

plt.show()